In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load silver layer tables
orders = spark.table("novacart_dev.silver.slv_orders")
order_items = spark.table("novacart_dev.silver.slv_order_items")
products = spark.table("novacart_dev.silver.slv_products")
customers = spark.table("novacart_dev.silver.slv_customers")
exchange = spark.table("novacart_dev.silver.slv_exchange_rates")

In [0]:
# Fact Table: Order Items (grain: one row per order line item)
# CURRENCY HANDLING:
# - orders.total_amount: Already in USD (currency field is just a label)
# - order_items.line_total: In product's local currency (needs conversion)
# - products.price: In product's local currency

fact_order_items = order_items \
    .join(orders, "order_id", "inner") \
    .join(products, "product_id", "left") \
    .join(customers, "customer_id", "left") \
    .join(
        exchange,
        (products.currency == exchange.currency_code),
        "left"
    ) \
    .select(
        # Keys
        F.col("order_item_id").alias("order_item_key"),
        F.col("order_id").alias("order_key"),
        F.col("product_id").alias("product_key"),
        F.col("customer_id").alias("customer_key"),
        F.col("order_date").alias("date_key"),
        
        # Measures
        "quantity",
        "unit_price",
        "line_total",
        # Convert line_total from product's local currency to USD
        F.round(
            F.col("line_total") * F.coalesce(F.col("exchange_rate_to_usd"), F.lit(1.0)),
            2
        ).alias("revenue_usd"),
        
        # Attributes (denormalized for query performance)
        "order_status_clean",
        orders.channel.alias("channel"),
        orders.country_code.alias("order_country"),
        "category",
        products.currency.alias("product_currency"),  # Changed from order.currency
        
        # Data Quality Flags - Order Items (6)
        order_items.dq_line_total_mismatch,
        order_items.dq_orphan_order,
        order_items.dq_orphan_product,
        order_items.dq_missing_order_item_id,
        order_items.dq_invalid_quantity,
        order_items.dq_invalid_price.alias("dq_invalid_item_price"),
        
        # Data Quality Flags - Orders (5)
        orders.dq_orphan_customer,
        orders.dq_missing_order_id,
        orders.dq_invalid_order_date,
        orders.dq_invalid_status,
        orders.dq_invalid_amount,
        
        # Data Quality Flags - Products (3)
        products.dq_missing_product_id,
        products.dq_missing_product_name,
        products.dq_invalid_price.alias("dq_invalid_product_price"),
        
        # Data Quality Flags - Customers (3)
        customers.dq_missing_customer_id,
        customers.dq_missing_email,
        customers.dq_invalid_channel.alias("dq_invalid_customer_channel"),
        
        # Data Quality Flags - Exchange Rates (3)
        exchange.dq_invalid_date.alias("dq_invalid_exchange_date"),
        exchange.dq_invalid_currency.alias("dq_invalid_exchange_currency"),
        exchange.dq_invalid_exchange,
        
        # Metadata
        F.current_timestamp().alias("etl_timestamp")
    )

In [0]:
# Write fact_order_items to Gold layer (partitioned by date for performance)
fact_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("date_key") \
    .saveAsTable("novacart_dev.gold.fact_order_items")

print("✓ fact_order_items written to novacart_dev.gold.fact_order_items")
print("  Partitioned by: date_key")

In [0]:
# Verification: Check revenue totals by currency
from pyspark.sql.functions import sum as _sum, count, round as _round

verification = spark.table("novacart_dev.gold.fact_order_items") \
    .groupBy("product_currency") \
    .agg(
        count("*").alias("num_items"),
        _round(_sum("line_total"), 2).alias("total_local_currency"),
        _round(_sum("revenue_usd"), 2).alias("total_usd")
    ) \
    .orderBy("product_currency")

display(verification)

print("\n✓ Currency conversion verified")
print("  - Product prices in local currency converted to USD")
print("  - Order totals already in USD (no conversion needed)")